# TMPC Paper Figure Workbench

Interactive Plotly figures for the conference paper. Each cell produces a publication-quality plot you can export with `fig.write_image("name.pdf")` or screenshot.

**Prereqs**:
```bash
pip install plotly kaleido jupyter
```
Run `python -m tmpc_platform.scripts.run_all --n 5000` first so `results/` exists.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
import numpy as np, pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from tmpc_platform.physics_engine import TMPCConfig, simulate_tmpc
from tmpc_platform.physics_engine.gaussian_beam import GaussianBeam, propagate_through_cell
from tmpc_platform.physics_engine.losses import compute_losses

RESULTS = '../results'
OUT     = '../results/paper_figs'
os.makedirs(OUT, exist_ok=True)
print('Plotly version:', __import__('plotly').__version__)

## 1. Rotatable 3D spot pattern

Pick a config from your Pareto front (or set one explicitly). Hold mouse + drag to rotate.

In [ ]:
# Use top Pareto config, or set explicitly below
try:
    par = pd.read_csv(f'{RESULTS}/optim/pareto.csv')
    best = par.iloc[par['obj_0'].idxmax()]  # max OPL on Pareto front
    cfg = TMPCConfig(N=int(best['N']), R_ring=float(best['R_ring']),
                     H=float(best['H']), R_t=float(best['R_t']),
                     R_s=float(best['R_s']), chord_skip=int(best['chord_skip']),
                     w0=float(best['w0']),
                     input_offset_z=float(best['input_offset_z']),
                     input_angle=float(best['input_angle']))
except Exception:
    cfg = TMPCConfig(N=14, R_ring=68, H=40, R_t=140, R_s=140,
                     chord_skip=5, w0=0.5, input_offset_z=1.5)
cfg.n_passes = 8 * cfg.N
res = simulate_tmpc(cfg)
print(f'N={cfg.N}, skip={cfg.chord_skip}, OPL={res.opl*1e-3:.2f} m, bounces={res.bounces}')

In [ ]:
sp = res.spot_pattern
th = np.linspace(0, 2*np.pi, cfg.N, endpoint=False)
fig = go.Figure()
fig.add_trace(go.Scatter3d(x=cfg.R_ring*np.cos(th), y=cfg.R_ring*np.sin(th),
                           z=np.zeros_like(th), mode='markers',
                           marker=dict(size=8, color='red'), name='mirrors'))
fig.add_trace(go.Scatter3d(x=sp[:,0], y=sp[:,1], z=sp[:,2],
                           mode='lines+markers',
                           line=dict(color='royalblue', width=3),
                           marker=dict(size=4, color=np.arange(len(sp)),
                                       colorscale='Viridis', showscale=True,
                                       colorbar=dict(title='bounce #')),
                           name='trajectory'))
fig.update_layout(scene=dict(aspectmode='data',
                             xaxis_title='x [mm]', yaxis_title='y [mm]',
                             zaxis_title='z [mm]'),
                  title=f'TMPC spot pattern  N={cfg.N}, skip={cfg.chord_skip}, OPL={res.opl*1e-3:.2f} m',
                  width=900, height=700)
fig.show()
# fig.write_image(f'{OUT}/fig1_spot_pattern.pdf')  # uncomment to export

## 2. Pareto front explorer (3D scatter with hover)

In [ ]:
par = pd.read_csv(f'{RESULTS}/optim/pareto.csv')
fig = px.scatter_3d(par, x='obj_0', y='obj_1', z='obj_2',
                    color='N', size_max=10,
                    hover_data=['N', 'chord_skip', 'R_ring', 'R_t', 'w0'],
                    labels={'obj_0':'OPL [m]', 'obj_1':'Throughput',
                            'obj_2':'-Clipping (higher=better)'})
fig.update_layout(title='NSGA-II Pareto front', width=900, height=650)
fig.show()

## 3. Dataset OPL vs throughput scatter (paper Fig. 2 candidate)

In [ ]:
df = pd.read_csv(f'{RESULTS}/dataset.csv')
fig = px.scatter(df, x='opl_m', y='throughput_full',
                 color='N', size='volume_utilisation', size_max=12,
                 hover_data=['chord_skip', 'R_t', 'R_ring', 'w0', 'stability_g'],
                 labels={'opl_m':'OPL [m]', 'throughput_full':'Throughput'})
fig.update_layout(title='Design-space coverage', width=950, height=600)
fig.show()

## 4. Per-bounce AOI for the best config

In [ ]:
fig = go.Figure(go.Scatter(x=np.arange(1, len(res.aoi)+1), y=res.aoi,
                           mode='lines+markers',
                           line=dict(color='darkorange', width=2)))
fig.add_hline(y=res.aoi.mean(), line=dict(color='black', dash='dash'),
              annotation_text=f'mean = {res.aoi.mean():.1f}°')
fig.update_layout(xaxis_title='bounce #', yaxis_title='AOI [deg]',
                  title=f'Per-bounce angle of incidence  (N={cfg.N}, skip={cfg.chord_skip})',
                  width=900, height=420)
fig.show()

## 5. Feature importance bar chart

In [ ]:
for tgt in ['opl_m', 'throughput_full', 'quality', 'stability_g']:
    p = f'{RESULTS}/explain/importance__{tgt}.csv'
    if not os.path.exists(p): continue
    imp = pd.read_csv(p, index_col=0).sort_values('importance')
    fig = go.Figure(go.Bar(x=imp['importance'], y=imp.index, orientation='h',
                           marker_color='seagreen'))
    fig.update_layout(title=f'Feature importance — {tgt}',
                      xaxis_title='|SHAP| or permutation importance',
                      width=750, height=380)
    fig.show()

## 6. Parameter sweep — OPL vs chord_skip at fixed N
Useful for the paper to show why chord_skip matters.

In [ ]:
from math import gcd
results = []
for N in [9, 12, 16]:
    for skip in range(1, N):
        c = TMPCConfig(N=N, R_ring=60, H=40, R_t=120, R_s=120,
                       chord_skip=skip, w0=0.5,
                       input_offset_z=1.0)
        c.n_passes = 8 * N
        r = simulate_tmpc(c)
        results.append({'N': N, 'skip': skip, 'opl_m': r.opl*1e-3,
                        'bounces': r.bounces, 'throughput': r.throughput,
                        'coprime': gcd(N, skip) == 1})
swp = pd.DataFrame(results)
fig = px.bar(swp, x='skip', y='opl_m', color='N', barmode='group',
             pattern_shape='coprime',
             labels={'opl_m':'OPL [m]', 'skip':'chord_skip'})
fig.update_layout(title='OPL vs chord_skip', width=900, height=450)
fig.show()